# GRASP One-Shot Designer

Design orderable GRASP Level -1 fragments for a binder that recognizes **one target RNA**.

Flow: RNA → GRASP modules → BsaI insertion into the configured Level −1 entry vector → BpiI assembly into the configured Level 0 acceptor.

1. Run **0 · Install** once  
2. Fill the forms below and run each cell top → bottom  

Code is hidden by default (Colab Forms). Cell ⋮ → **Form → Hide code** if needed.


In [ ]:
#@title 0 · Install (PyPI) { display-mode: "form" }
#@markdown Installs everything from PyPI. No GitHub token needed. Re-run if imports fail after a runtime restart.

%pip install -q -U --force-reinstall "grasp-library-designer>=0.1.10"
import importlib, sys
for _m in [m for m in list(sys.modules) if m == 'grasp_library' or m.startswith('grasp_library.')]:
    del sys.modules[_m]


import importlib
import grasp_library
from importlib.metadata import version

print("grasp-library-designer", version("grasp-library-designer"))
print("import ok:", grasp_library.__name__)
from grasp_library import notebook_ui as _ui
print("kazusa_codon_reminder:", hasattr(_ui, "kazusa_codon_reminder"))


In [ ]:
#@title 1 · Settings { display-mode: "form" }
#@markdown Choose organism, synthesis, ligation, target RNA, and cloning-level overhangs — then run this cell.
#@markdown **Codon tables:** browse [Kazusa CUTG](https://www.kazusa.or.jp/codon/) (search → copy the `species=` accession from the URL). Built-ins cover common hosts; otherwise choose *Fetch from Kazusa* or *Upload your own*.

target_rna = "UUACACGUG" #@param {type:"string"}
organism = "Escherichia coli (Kazusa)" #@param ["Escherichia coli (Kazusa)", "Saccharomyces cerevisiae (Kazusa)", "Homo sapiens (Kazusa)", "Euglena gracilis nuclear (Kazusa)", "Chlamydomonas reinhardtii nuclear (Kazusa)", "Chlamydomonas reinhardtii chloroplast (Kazusa)", "Fetch from Kazusa by species ID", "Upload your own codon table"]
kazusa_species_id = "" #@param {type:"string"}
genetic_code = 1 #@param {type:"integer"}
synthesis_vendor = "Twist · Standard gene guidelines" #@param ["Twist · Express / Low complexity", "Twist · Standard gene guidelines", "Twist · Complex Genes tolerant", "IDT · gBlocks / eBlocks conservative", "Generic · conservative (default)"]
assembly_enzyme = "GRASP default · BsaI + BpiI + BsmBI" #@param ["GRASP default · BsaI + BpiI + BsmBI", "BsaI (GGTCTC)", "BpiI / BbsI (GAAGAC)", "BsmBI / Esp3I (CGTCTC)", "None (no enzyme filter)"]
#@markdown `assembly_enzyme` controls internal-site domestication; the cloning enzymes remain BsaI then BpiI.
ligation_table = "GRASP Level 0 proxy · BbsI-HF + T4 · 37↔16 °C cycling (Pryor 2020)" #@param ["GRASP Level 0 proxy · BbsI-HF + T4 · 37↔16 °C cycling (Pryor 2020)", "T4 ligase only · 18 h · 25 °C (Potapov 2018; validated cycling proxy)", "T4 ligase only · 1 h · 25 °C (Potapov 2018)", "T4 ligase only · 1 h · 37 °C (Potapov 2018)", "T4 ligase only · 18 h · 37 °C (Potapov 2018)"]
#@markdown **Golden Gate overhangs:** enter every physical sticky end 5′→3′. Defaults are from the deposited GRASP toolbox.
level_minus1_5prime_overhang = "ACAT" #@param {type:"string"}
level_minus1_3prime_overhang = "ACAA" #@param {type:"string"}
level0_5prime_overhang = "CTCA" #@param {type:"string"}
level0_3prime_overhang = "CTCG" #@param {type:"string"}
level1_5prime_overhang = "GGAG" #@param {type:"string"}
level1_3prime_overhang = "AGCG" #@param {type:"string"}
optimize_depth = 2000 #@param {type:"integer"}

from grasp_library import build_default_config, materialize_project
from grasp_library.colab_forms import apply_form_settings
from grasp_library import notebook_ui as ui

PROJECT_DIR = materialize_project()
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_ROOT = PROJECT_DIR / "output" / "oneshot"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = build_default_config(INPUT_DIR)
CONFIG["project_name"] = "GRASP_oneshot_colab"

if hasattr(ui, "kazusa_codon_reminder"):
    ui.kazusa_codon_reminder()
else:
    ui.note(
        "Package is outdated (missing Kazusa helper). "
        "Re-run <b>0 · Install</b>, then Runtime → Restart session, then Settings again."
    )

applied = apply_form_settings(
    CONFIG,
    organism=organism,
    genetic_code=int(genetic_code),
    target_rna=target_rna,
    synthesis_vendor=synthesis_vendor,
    assembly_enzyme=assembly_enzyme,
    ligation_table=ligation_table,
    level_minus1_5prime_overhang=level_minus1_5prime_overhang,
    level_minus1_3prime_overhang=level_minus1_3prime_overhang,
    level0_5prime_overhang=level0_5prime_overhang,
    level0_3prime_overhang=level0_3prime_overhang,
    level1_5prime_overhang=level1_5prime_overhang,
    level1_3prime_overhang=level1_3prime_overhang,
    optimize_depth=int(optimize_depth),
    kazusa_species_id=kazusa_species_id,
    overhang_redesign=False,
)
CONFIG = applied["config"]
CODON_DATA = applied["codon_data"]
ONESHOT = None
ORGANISM_LABEL = applied.get("meta", {}).get("organism", organism)

ui.status(
    f"Target <b>{CONFIG['target_rna']}</b> · organism <b>{ORGANISM_LABEL}</b> · "
    f"genetic code <b>{CONFIG['genetic_code']}</b> · "
    f"depth <b>{CONFIG['optimizer']['iterations_per_part']:,}</b> · "
    f"vendor <b>{CONFIG['synthesis_vendor']}</b><br/>"
    f"Level −1 5′/3′ <code>{level_minus1_5prime_overhang}/{level_minus1_3prime_overhang}</code> · "
    f"Level 0 5′/3′ <code>{level0_5prime_overhang}/{level0_3prime_overhang}</code> · "
    f"Level 1 5′/3′ <code>{level1_5prime_overhang}/{level1_3prime_overhang}</code><br/>"
    f"Ligation model: <b>{CONFIG['ligation']['table_name']}</b><br/>"
    f"Translation QC uses this organism codon table · Project → <code>{PROJECT_DIR}</code>"
)


In [ ]:
#@title 2 · Preview binder protein { display-mode: "form" }

from grasp_library import describe_binder
from grasp_library import notebook_ui as ui

info = describe_binder(CONFIG["target_rna"])
ui.status(
    f"<b>{info['target_rna']}</b> · PPR <code>{info['ppr_code']}</code> · "
    f"<b>{info['aa_length']}</b> aa · <b>{info['cds_length']}</b> nt · "
    f"architecture <b>{len(CONFIG['target_rna'])}S</b> · fixed five-part Level 0 blocks"
)
print(info["aa_sequence"])


In [ ]:
#@title 3 · Design oligos { display-mode: "form" }
#@markdown Re-run after changing settings. Files appear under `grasp_library_project/output/oneshot/`.

RUN_ONESHOT = True #@param {type:"boolean"}
SEED = 42 #@param {type:"integer"}

import random
import numpy as np
from IPython.display import display
from grasp_library import run_oneshot_design, sanitize_rna_name
from grasp_library import notebook_ui as ui

random.seed(int(SEED))
np.random.seed(int(SEED))

ONESHOT = None
if not RUN_ONESHOT:
    ui.note("RUN_ONESHOT is off.")
elif not CODON_DATA:
    ui.note("Run the Settings cell first.")
else:
    rna = sanitize_rna_name(CONFIG["target_rna"])
    out_dir = OUTPUT_ROOT / rna
    ONESHOT = run_oneshot_design(
        target_rna=CONFIG["target_rna"],
        codon_data=CODON_DATA,
        config=CONFIG,
        output_dir=out_dir,
        seed=int(SEED),
        log=print,
    )
    display(ONESHOT["assembly_plan"][
        ["assembly_group", "assembly_order", "part_id", "module_release_oh5", "module_release_oh3"]
    ])
    cols = [c for c in [
        "order_fragment_id", "order_quantity", "oligo_length", "oligo_gc",
        "qc_status", "hard_constraints_passed", "order_sequence_5to3",
    ] if c in ONESHOT["oligos"].columns]
    display(ONESHOT["oligos"][cols])
    asm = ONESHOT["assembled"]
    ui.status(
        f"Translation verified: <b>{asm['translation_verified']}</b> · "
        f"codon table OK: <b>{asm.get('codon_table_ok')}</b> · "
        f"code <b>{asm.get('genetic_code')}</b> · "
        f"Configured BsaI entry + BpiI Level 0 + PPR block-chain checks <b>passed</b> · "
        f"order file → <code>{ONESHOT['order_csv']}</code>"
    )


In [ ]:
#@title 4 · Export Excel { display-mode: "form" }

import pandas as pd
from grasp_library import notebook_ui as ui

if ONESHOT is None:
    ui.note("Run Design first.")
else:
    out = ONESHOT["output_dir"]
    xlsx = out / f"oneshot_{ONESHOT['target_rna']}.xlsx"
    with pd.ExcelWriter(xlsx) as writer:
        pd.DataFrame([ONESHOT["binder"]]).to_excel(writer, sheet_name="binder", index=False)
        ONESHOT["assembly_plan"].to_excel(writer, sheet_name="assembly_plan", index=False)
        ONESHOT["oligos"].to_excel(writer, sheet_name="oligos", index=False)
        ONESHOT["level0_assemblies"].to_excel(writer, sheet_name="level0_blocks", index=False)
        pd.DataFrame([ONESHOT["ppr_block_chain"]]).to_excel(writer, sheet_name="ppr_chain", index=False)
        pd.DataFrame([ONESHOT["summary"]]).to_excel(writer, sheet_name="summary", index=False)
    ui.status(f"Wrote <code>{xlsx}</code>")
    if "google.colab" in __import__("sys").modules:
        from google.colab import files
        files.download(str(xlsx))
    for p in sorted(out.glob("*")):
        if p.is_file():
            print(p.name)


## Notes

| Step | What happens |
|---|---|
| Install | `pip install grasp-library-designer` from PyPI |
| RNA → modules | GAP selects the target-specific GRASP Level -1 parts |
| Optimize | Each selected module is synonymously optimized |
| Entry clone | BsaI insert is built for the configured Level -1 vector interfaces |
| Level 0 | BpiI release payloads assemble in the configured Level 0 acceptor |
| Level 1 | Configured directional interfaces are checked for the PPR block chain; accessory modules and acceptor sequence are not simulated unless supplied |
| Export | Deduplicated, orderable dsDNA fragments plus assembly plans |

For the **42-module combinatorial library**, open `grasp_library_designer.ipynb`.

Package: https://pypi.org/project/grasp-library-designer/
